In [73]:
from pathlib import Path
import gc
import time

import numpy as np
import pandas as pd

from nuscenes.nuscenes import NuScenes
from nuscenes.map_expansion.map_api import NuScenesMap

from bst_bev.maps.bst_map_index import BSTMapIndex
from bst_bev.maps.hierarchical_bst_map_index import (
    HierarchicalBSTMapIndex,
)

from bst_bev.query.fixed_zoom import (
    FixedZoomRegionCover,
)

from bst_bev.query.direct_hierarchical import (
    DirectHierarchicalRegionCover,
)

In [74]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATAROOT = PROJECT_ROOT / "data" / "nuscenes"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RAW_OUTPUT = (
    OUTPUT_DIR
    / "step3d_hierarchical_all_samples.csv"
)

SUMMARY_OUTPUT = (
    OUTPUT_DIR
    / "step3d_hierarchical_summary.csv"
)

STORAGE_OUTPUT = (
    OUTPUT_DIR
    / "step3d_hierarchical_storage.csv"
)

STORAGE_SUMMARY_OUTPUT = (
    OUTPUT_DIR
    / "step3d_hierarchical_storage_summary.csv"
)


ZOOMS = [23, 24]

MIN_HIERARCHICAL_ZOOM = 19
MAX_HIERARCHICAL_ZOOM = 24

RADIUS_M = 30.0

INDEX_SPACING_M = 1.0

# Repeat small timing operations and take their median.
# This is much more stable than timing once.
TIMING_REPEATS = 5


MAP_LAYERS = (
    "lane_divider",
    "road_divider",
    "ped_crossing",
)

In [75]:
nusc = NuScenes(
    version="v1.0-mini",
    dataroot=str(DATAROOT),
    verbose=False,
)

print("Scenes:", len(nusc.scene))
print("Samples:", len(nusc.sample))

Scenes: 10
Samples: 404


In [76]:
def get_sample_context(
    nusc,
    sample,
):
    scene = nusc.get(
        "scene",
        sample["scene_token"],
    )

    log = nusc.get(
        "log",
        scene["log_token"],
    )

    location = log["location"]

    lidar = nusc.get(
        "sample_data",
        sample["data"]["LIDAR_TOP"],
    )

    ego_pose = nusc.get(
        "ego_pose",
        lidar["ego_pose_token"],
    )

    ego_x, ego_y, _ = (
        ego_pose["translation"]
    )

    return {
        "scene": scene["name"],
        "location": location,
        "ego_x": float(ego_x),
        "ego_y": float(ego_y),
    }

In [77]:
sample_contexts = []

for sample in nusc.sample:

    context = get_sample_context(
        nusc,
        sample,
    )

    sample_contexts.append({
        "sample_token": sample["token"],
        "timestamp": sample["timestamp"],
        **context,
    })


samples_df = pd.DataFrame(
    sample_contexts
)

samples_df.head()

,sample_token,timestamp,scene,location,ego_x,ego_y
0,ca9a282c9e77460f8360f564131a8af5,1532402927647951,scene-0061,singapore-onenorth,411.303935,1180.890379
1,39586f9d59004284a7114a68825e8eec,1532402928147847,scene-0061,singapore-onenorth,409.743152,1176.676973
2,356d81f38dd9473ba590f39e266f54e5,1532402928698048,scene-0061,singapore-onenorth,408.101879,1172.151578
3,e0845f5322254dafadbbed75aaa07969,1532402929197353,scene-0061,singapore-onenorth,406.726006,1168.211287
4,c923fe08b2ff4e27975d2bf30934383b,1532402929697797,scene-0061,singapore-onenorth,405.409406,1164.354227


In [78]:
locations = sorted(
    samples_df["location"].unique()
)

nusc_maps = {
    location: NuScenesMap(
        dataroot=str(DATAROOT),
        map_name=location,
    )
    for location in locations
}

for location in locations:
    print(location)

boston-seaport
singapore-hollandvillage
singapore-onenorth
singapore-queenstown


In [79]:
def get_ground_truth_tokens(
    nusc_map,
    ego_x,
    ego_y,
    radius=RADIUS_M,
):
    patch = (
        ego_x - radius,
        ego_y - radius,
        ego_x + radius,
        ego_y + radius,
    )

    records = (
        nusc_map.get_records_in_patch(
            patch,
            list(MAP_LAYERS),
            mode="intersect",
        )
    )

    return {
        (layer, token)
        for layer, tokens
        in records.items()
        for token in tokens
    }

In [80]:
ground_truth_cache = {}


for _, row in samples_df.iterrows():

    gt_tokens = (
        get_ground_truth_tokens(
            nusc_map=nusc_maps[
                row["location"]
            ],
            ego_x=row["ego_x"],
            ego_y=row["ego_y"],
        )
    )

    ground_truth_cache[
        row["sample_token"]
    ] = gt_tokens


print(
    "Ground truth cached for",
    len(ground_truth_cache),
    "samples",
)

Ground truth cached for 404 samples


In [81]:
def median_runtime_ms(
    function,
    repeats=TIMING_REPEATS,
):
    timings = []

    for _ in range(repeats):

        start = time.perf_counter()

        function()

        elapsed_ms = (
            time.perf_counter()
            - start
        ) * 1000.0

        timings.append(
            elapsed_ms
        )

    return float(
        np.median(timings)
    )

In [82]:
def evaluate_sample(
    *,
    row,
    gt_tokens,

    fixed_index,
    hierarchical_index,

    fixed_cover_generator,
    hierarchical_cover_generator,
):
    ego_x = row["ego_x"]
    ego_y = row["ego_y"]

    # ======================================================
    # Construct covers once for correctness evaluation
    # ======================================================

    fixed_cover = (
        fixed_cover_generator.cover(
            center_x=ego_x,
            center_y=ego_y,
            radius=RADIUS_M,
        )
    )

    hierarchical_cover = (
        hierarchical_cover_generator.cover(
            center_x=ego_x,
            center_y=ego_y,
            radius=RADIUS_M,
        )
    )

    # ======================================================
    # Candidate retrieval
    # ======================================================

    fixed_tokens = (
        fixed_index.query_cell_tokens(
            fixed_cover.keys
        )
    )

    hierarchical_result = (
        hierarchical_index.query_cells(
            hierarchical_cover.cells
        )
    )

    hierarchical_tokens = (
        hierarchical_result.tokens
    )

    # ======================================================
    # Fixed accuracy
    # ======================================================

    fixed_tp = len(
        fixed_tokens & gt_tokens
    )

    fixed_fp = len(
        fixed_tokens - gt_tokens
    )

    fixed_fn = len(
        gt_tokens - fixed_tokens
    )

    fixed_precision = (
        fixed_tp
        / (fixed_tp + fixed_fp)
        if fixed_tp + fixed_fp
        else np.nan
    )

    fixed_recall = (
        fixed_tp
        / (fixed_tp + fixed_fn)
        if fixed_tp + fixed_fn
        else np.nan
    )

    # ======================================================
    # Hierarchical accuracy
    # ======================================================

    hierarchical_tp = len(
        hierarchical_tokens
        & gt_tokens
    )

    hierarchical_fp = len(
        hierarchical_tokens
        - gt_tokens
    )

    hierarchical_fn = len(
        gt_tokens
        - hierarchical_tokens
    )

    hierarchical_precision = (
        hierarchical_tp
        / (
            hierarchical_tp
            + hierarchical_fp
        )
        if (
            hierarchical_tp
            + hierarchical_fp
        )
        else np.nan
    )

    hierarchical_recall = (
        hierarchical_tp
        / (
            hierarchical_tp
            + hierarchical_fn
        )
        if (
            hierarchical_tp
            + hierarchical_fn
        )
        else np.nan
    )

    # ======================================================
    # Candidate equivalence
    # ======================================================

    candidate_intersection = (
        fixed_tokens
        & hierarchical_tokens
    )

    candidate_union = (
        fixed_tokens
        | hierarchical_tokens
    )

    candidate_jaccard = (
        len(candidate_intersection)
        / len(candidate_union)
        if candidate_union
        else 1.0
    )

    same_candidates = (
        fixed_tokens
        == hierarchical_tokens
    )

    # ======================================================
    # Timing
    # ======================================================

    fixed_cover_time_ms = (
        median_runtime_ms(
            lambda:
            fixed_cover_generator.cover(
                center_x=ego_x,
                center_y=ego_y,
                radius=RADIUS_M,
            )
        )
    )

    fixed_lookup_time_ms = (
        median_runtime_ms(
            lambda:
            fixed_index.query_cell_tokens(
                fixed_cover.keys
            )
        )
    )

    hierarchical_cover_time_ms = (
        median_runtime_ms(
            lambda:
            hierarchical_cover_generator.cover(
                center_x=ego_x,
                center_y=ego_y,
                radius=RADIUS_M,
            )
        )
    )

    hierarchical_lookup_time_ms = (
        median_runtime_ms(
            lambda:
            hierarchical_index.query_cells(
                hierarchical_cover.cells
            )
        )
    )

    fixed_total_time_ms = (
        fixed_cover_time_ms
        + fixed_lookup_time_ms
    )

    hierarchical_total_time_ms = (
        hierarchical_cover_time_ms
        + hierarchical_lookup_time_ms
    )

    query_speedup = (
        fixed_total_time_ms
        / hierarchical_total_time_ms
        if hierarchical_total_time_ms > 0
        else np.nan
    )

    # ======================================================
    # Result
    # ======================================================

    return {
        # Region representation.
        "fixed_cells":
            fixed_cover.cell_count,

        "hierarchical_cells":
            hierarchical_cover.hierarchical_cell_count,

        "compression_ratio":
            hierarchical_cover.compression_ratio,

        "reduction_fraction":
            hierarchical_cover.reduction_fraction,

        # Candidates.
        "fixed_candidates":
            len(fixed_tokens),

        "hierarchical_candidates":
            len(hierarchical_tokens),

        "same_candidates":
            same_candidates,

        "candidate_jaccard":
            candidate_jaccard,

        "fixed_only_count":
            len(
                fixed_tokens
                - hierarchical_tokens
            ),

        "hierarchical_only_count":
            len(
                hierarchical_tokens
                - fixed_tokens
            ),

        # Ground truth.
        "ground_truth":
            len(gt_tokens),

        # Fixed metrics.
        "fixed_tp":
            fixed_tp,

        "fixed_fp":
            fixed_fp,

        "fixed_fn":
            fixed_fn,

        "fixed_precision":
            fixed_precision,

        "fixed_recall":
            fixed_recall,

        # Hierarchical metrics.
        "hierarchical_tp":
            hierarchical_tp,

        "hierarchical_fp":
            hierarchical_fp,

        "hierarchical_fn":
            hierarchical_fn,

        "hierarchical_precision":
            hierarchical_precision,

        "hierarchical_recall":
            hierarchical_recall,

        # Fixed timing.
        "fixed_cover_time_ms":
            fixed_cover_time_ms,

        "fixed_lookup_time_ms":
            fixed_lookup_time_ms,

        "fixed_total_time_ms":
            fixed_total_time_ms,

        # Hierarchical timing.
        "hierarchical_cover_time_ms":
            hierarchical_cover_time_ms,

        "hierarchical_lookup_time_ms":
            hierarchical_lookup_time_ms,

        "hierarchical_total_time_ms":
            hierarchical_total_time_ms,

        # Comparison.
        "query_speedup":
            query_speedup,
    }

In [83]:
def fixed_membership_count(
    index,
):
    return sum(
        len(primitives)
        for primitives
        in index.index.values()
    )

In [84]:
all_results = []
storage_results = []

experiment_start = (
    time.perf_counter()
)


for location in locations:

    print()
    print("=" * 70)
    print(location)
    print("=" * 70)

    nusc_map = nusc_maps[
        location
    ]

    location_samples = (
        samples_df[
            samples_df["location"]
            == location
        ]
    )

    # ======================================================
    # BUILD HIERARCHICAL INDEX ONCE
    # ======================================================

    print(
        "Building hierarchical index..."
    )

    hierarchical_build_start = (
        time.perf_counter()
    )

    hierarchical_index = (
        HierarchicalBSTMapIndex(
            nusc_map=nusc_map,
            location=location,

            min_zoom=(
                MIN_HIERARCHICAL_ZOOM
            ),

            max_zoom=(
                MAX_HIERARCHICAL_ZOOM
            ),

            index_spacing_m=(
                INDEX_SPACING_M
            ),
        )
    )

    hierarchical_index.build()

    hierarchical_build_time_s = (
        time.perf_counter()
        - hierarchical_build_start
    )

    print(
        "  hierarchical keys:",
        f"{hierarchical_index.key_count:,}",
    )

    print(
        "  hierarchical memberships:",
        f"{hierarchical_index.membership_count:,}",
    )

    print(
        "  build time:",
        f"{hierarchical_build_time_s:.3f}s",
    )

    # ======================================================
    # EVALUATE EACH FIXED ZOOM
    # ======================================================

    for zoom in ZOOMS:

        print()
        print(
            f"  Zoom {zoom}"
        )

        # ----------------------------------------------
        # Fixed index
        # ----------------------------------------------

        fixed_build_start = (
            time.perf_counter()
        )

        fixed_index = BSTMapIndex(
            nusc_map=nusc_map,
            location=location,
            zoom=zoom,

            index_spacing_m=(
                INDEX_SPACING_M
            ),

            adaptive_index_spacing=False,
        )

        fixed_index.build()

        fixed_build_time_s = (
            time.perf_counter()
            - fixed_build_start
        )

        fixed_memberships = (
            fixed_membership_count(
                fixed_index
            )
        )

        # ----------------------------------------------
        # Storage metrics
        # ----------------------------------------------

        storage_results.append({
            "location":
                location,

            "zoom":
                zoom,

            "fixed_keys":
                len(fixed_index),

            "fixed_memberships":
                fixed_memberships,

            "fixed_build_time_s":
                fixed_build_time_s,

            "hierarchical_keys":
                hierarchical_index.key_count,

            "hierarchical_memberships":
                hierarchical_index.membership_count,

            "hierarchical_build_time_s":
                hierarchical_build_time_s,

            "key_multiplier": (
                hierarchical_index.key_count
                / len(fixed_index)
                if len(fixed_index)
                else np.nan
            ),

            "membership_multiplier": (
                hierarchical_index.membership_count
                / fixed_memberships
                if fixed_memberships
                else np.nan
            ),
        })

        # ----------------------------------------------
        # Query generators
        # ----------------------------------------------

        fixed_cover_generator = (
            FixedZoomRegionCover(
                location=location,
                zoom=zoom,
            )
        )

        hierarchical_cover_generator = (
            DirectHierarchicalRegionCover(
                location=location,
                zoom=zoom,
            )
        )

        # ----------------------------------------------
        # Warm-up
        # ----------------------------------------------

        if len(location_samples):

            warmup_row = (
                location_samples.iloc[0]
            )

            warmup_fixed = (
                fixed_cover_generator.cover(
                    center_x=(
                        warmup_row["ego_x"]
                    ),
                    center_y=(
                        warmup_row["ego_y"]
                    ),
                    radius=RADIUS_M,
                )
            )

            fixed_index.query_cell_tokens(
                warmup_fixed.keys
            )

            warmup_hier = (
                hierarchical_cover_generator.cover(
                    center_x=(
                        warmup_row["ego_x"]
                    ),
                    center_y=(
                        warmup_row["ego_y"]
                    ),
                    radius=RADIUS_M,
                )
            )

            hierarchical_index.query_cells(
                warmup_hier.cells
            )

        # ----------------------------------------------
        # All samples in this map
        # ----------------------------------------------

        for _, row in (
            location_samples.iterrows()
        ):

            gt_tokens = (
                ground_truth_cache[
                    row["sample_token"]
                ]
            )

            metrics = evaluate_sample(
                row=row,
                gt_tokens=gt_tokens,

                fixed_index=fixed_index,

                hierarchical_index=(
                    hierarchical_index
                ),

                fixed_cover_generator=(
                    fixed_cover_generator
                ),

                hierarchical_cover_generator=(
                    hierarchical_cover_generator
                ),
            )

            all_results.append({
                "zoom":
                    zoom,

                "scene":
                    row["scene"],

                "sample_token":
                    row["sample_token"],

                "timestamp":
                    row["timestamp"],

                "location":
                    location,

                "ego_x":
                    row["ego_x"],

                "ego_y":
                    row["ego_y"],

                **metrics,
            })

        # ----------------------------------------------
        # Checkpoint
        # ----------------------------------------------

        results_df = pd.DataFrame(
            all_results
        )

        results_df.to_csv(
            RAW_OUTPUT,
            index=False,
        )

        storage_df = pd.DataFrame(
            storage_results
        )

        storage_df.to_csv(
            STORAGE_OUTPUT,
            index=False,
        )

        print(
            f"    completed "
            f"{len(location_samples)} samples"
        )

        print(
            f"    fixed cells mean: "
            f"{results_df[
                results_df['zoom'] == zoom
            ]['fixed_cells'].mean():.1f}"
        )

        del fixed_index
        gc.collect()

    # Hierarchical index has now served every
    # supported zoom/sample for this location.
    del hierarchical_index
    gc.collect()


experiment_time_s = (
    time.perf_counter()
    - experiment_start
)

print()
print(
    "Experiment complete in",
    f"{experiment_time_s / 60:.2f}",
    "minutes",
)


boston-seaport
Building hierarchical index...
  hierarchical keys: 297,437
  hierarchical memberships: 466,330
  build time: 2.740s

  Zoom 23
    completed 163 samples
    fixed cells mean: 324.0

  Zoom 24
    completed 163 samples
    fixed cells mean: 1224.8

singapore-hollandvillage
Building hierarchical index...
  hierarchical keys: 70,414
  hierarchical memberships: 100,162
  build time: 0.384s

  Zoom 23
    completed 121 samples
    fixed cells mean: 263.9

  Zoom 24
    completed 121 samples
    fixed cells mean: 994.0

singapore-onenorth
Building hierarchical index...
  hierarchical keys: 103,158
  hierarchical memberships: 148,255
  build time: 0.699s

  Zoom 23
    completed 39 samples
    fixed cells mean: 254.3

  Zoom 24
    completed 39 samples
    fixed cells mean: 956.4

singapore-queenstown
Building hierarchical index...
  hierarchical keys: 81,571
  hierarchical memberships: 110,365
  build time: 0.639s

  Zoom 23
    completed 81 samples
    fixed cells mean: 239

In [85]:
zoom_rows = results_df[
    results_df["zoom"] == zoom
]

print(
    "    mean fixed cells:",
    f"{zoom_rows['fixed_cells'].mean():.1f}",
)

    mean fixed cells: 901.3


In [86]:
print(
    "Rows:",
    len(results_df),
)

print(
    "Unique samples:",
    results_df[
        "sample_token"
    ].nunique(),
)

Rows: 808
Unique samples: 404


In [87]:
agreement = (
    results_df
    .groupby("zoom")
    .agg(
        samples=(
            "sample_token",
            "count",
        ),

        exact_candidate_matches=(
            "same_candidates",
            "sum",
        ),

        agreement_rate=(
            "same_candidates",
            "mean",
        ),

        min_jaccard=(
            "candidate_jaccard",
            "min",
        ),

        mean_jaccard=(
            "candidate_jaccard",
            "mean",
        ),

        total_fixed_only=(
            "fixed_only_count",
            "sum",
        ),

        total_hierarchical_only=(
            "hierarchical_only_count",
            "sum",
        ),
    )
    .reset_index()
)

agreement

,zoom,samples,exact_candidate_matches,agreement_rate,min_jaccard,mean_jaccard,total_fixed_only,total_hierarchical_only
0,23,404,404,1.0,1.0,1.0,0,0
1,24,404,404,1.0,1.0,1.0,0,0


In [88]:
mismatches = results_df[
    ~results_df["same_candidates"]
]

print(
    "Candidate mismatches:",
    len(mismatches),
)

mismatches[
    [
        "zoom",
        "scene",
        "sample_token",
        "location",

        "fixed_cells",
        "hierarchical_cells",

        "fixed_candidates",
        "hierarchical_candidates",

        "candidate_jaccard",

        "fixed_only_count",
        "hierarchical_only_count",
    ]
].head(30)

Candidate mismatches: 0


,zoom,scene,sample_token,location,fixed_cells,hierarchical_cells,fixed_candidates,hierarchical_candidates,candidate_jaccard,fixed_only_count,hierarchical_only_count


In [89]:
summary = (
    results_df
    .groupby("zoom")
    .agg(
        samples=(
            "sample_token",
            "count",
        ),

        # --------------------------------
        # Equivalence
        # --------------------------------

        exact_candidate_matches=(
            "same_candidates",
            "sum",
        ),

        agreement_rate=(
            "same_candidates",
            "mean",
        ),

        min_candidate_jaccard=(
            "candidate_jaccard",
            "min",
        ),

        # --------------------------------
        # Region representation
        # --------------------------------

        mean_fixed_cells=(
            "fixed_cells",
            "mean",
        ),

        median_fixed_cells=(
            "fixed_cells",
            "median",
        ),

        mean_hierarchical_cells=(
            "hierarchical_cells",
            "mean",
        ),

        median_hierarchical_cells=(
            "hierarchical_cells",
            "median",
        ),

        mean_compression_ratio=(
            "compression_ratio",
            "mean",
        ),

        median_compression_ratio=(
            "compression_ratio",
            "median",
        ),

        mean_reduction_fraction=(
            "reduction_fraction",
            "mean",
        ),

        # --------------------------------
        # Fixed accuracy
        # --------------------------------

        fixed_tp=(
            "fixed_tp",
            "sum",
        ),

        fixed_fp=(
            "fixed_fp",
            "sum",
        ),

        fixed_fn=(
            "fixed_fn",
            "sum",
        ),

        # --------------------------------
        # Hierarchical accuracy
        # --------------------------------

        hierarchical_tp=(
            "hierarchical_tp",
            "sum",
        ),

        hierarchical_fp=(
            "hierarchical_fp",
            "sum",
        ),

        hierarchical_fn=(
            "hierarchical_fn",
            "sum",
        ),

        # --------------------------------
        # Fixed latency
        # --------------------------------

        mean_fixed_ms=(
            "fixed_total_time_ms",
            "mean",
        ),

        median_fixed_ms=(
            "fixed_total_time_ms",
            "median",
        ),

        p95_fixed_ms=(
            "fixed_total_time_ms",
            lambda x:
                x.quantile(0.95),
        ),

        # --------------------------------
        # Hierarchical latency
        # --------------------------------

        mean_hierarchical_ms=(
            "hierarchical_total_time_ms",
            "mean",
        ),

        median_hierarchical_ms=(
            "hierarchical_total_time_ms",
            "median",
        ),

        p95_hierarchical_ms=(
            "hierarchical_total_time_ms",
            lambda x:
                x.quantile(0.95),
        ),

        # --------------------------------
        # Cover latency
        # --------------------------------

        median_fixed_cover_ms=(
            "fixed_cover_time_ms",
            "median",
        ),

        median_hierarchical_cover_ms=(
            "hierarchical_cover_time_ms",
            "median",
        ),

        # --------------------------------
        # Lookup latency
        # --------------------------------

        median_fixed_lookup_ms=(
            "fixed_lookup_time_ms",
            "median",
        ),

        median_hierarchical_lookup_ms=(
            "hierarchical_lookup_time_ms",
            "median",
        ),

        # --------------------------------
        # Speedup
        # --------------------------------

        mean_query_speedup=(
            "query_speedup",
            "mean",
        ),

        median_query_speedup=(
            "query_speedup",
            "median",
        ),
    )
    .reset_index()
)

In [90]:
summary["fixed_precision"] = (
    summary["fixed_tp"]
    / (
        summary["fixed_tp"]
        + summary["fixed_fp"]
    )
)

summary["fixed_recall"] = (
    summary["fixed_tp"]
    / (
        summary["fixed_tp"]
        + summary["fixed_fn"]
    )
)


summary[
    "hierarchical_precision"
] = (
    summary["hierarchical_tp"]
    / (
        summary["hierarchical_tp"]
        + summary["hierarchical_fp"]
    )
)

summary[
    "hierarchical_recall"
] = (
    summary["hierarchical_tp"]
    / (
        summary["hierarchical_tp"]
        + summary["hierarchical_fn"]
    )
)

In [91]:
summary = summary[
    [
        "zoom",

        "samples",

        "agreement_rate",
        "min_candidate_jaccard",

        "fixed_precision",
        "fixed_recall",

        "hierarchical_precision",
        "hierarchical_recall",

        "mean_fixed_cells",
        "mean_hierarchical_cells",

        "mean_compression_ratio",
        "median_compression_ratio",
        "mean_reduction_fraction",

        "median_fixed_cover_ms",
        "median_hierarchical_cover_ms",

        "median_fixed_lookup_ms",
        "median_hierarchical_lookup_ms",

        "median_fixed_ms",
        "median_hierarchical_ms",

        "p95_fixed_ms",
        "p95_hierarchical_ms",

        "mean_query_speedup",
        "median_query_speedup",
    ]
]

summary

,zoom,samples,agreement_rate,min_candidate_jaccard,fixed_precision,fixed_recall,hierarchical_precision,hierarchical_recall,mean_fixed_cells,mean_hierarchical_cells,...,median_fixed_cover_ms,median_hierarchical_cover_ms,median_fixed_lookup_ms,median_hierarchical_lookup_ms,median_fixed_ms,median_hierarchical_ms,p95_fixed_ms,p95_hierarchical_ms,mean_query_speedup,median_query_speedup
0,23,404,1.0,1.0,0.939450,1.0,0.939450,1.0,239.913366,16.242574,...,0.256688,0.037375,0.726250,0.04402,0.983875,0.081458,1.656546,0.140460,15.119999,13.708403
1,24,404,1.0,1.0,0.959179,1.0,0.959179,1.0,901.287129,24.396040,...,0.984813,0.048875,2.592396,0.06725,3.580105,0.116230,6.299769,0.166528,40.384001,39.976927


In [92]:
summary.to_csv(
    SUMMARY_OUTPUT,
    index=False,
)

In [93]:
storage_summary = (
    storage_df
    .groupby("zoom")
    .agg(
        fixed_keys=(
            "fixed_keys",
            "sum",
        ),

        fixed_memberships=(
            "fixed_memberships",
            "sum",
        ),

        fixed_build_time_s=(
            "fixed_build_time_s",
            "sum",
        ),

        hierarchical_keys=(
            "hierarchical_keys",
            "sum",
        ),

        hierarchical_memberships=(
            "hierarchical_memberships",
            "sum",
        ),

        hierarchical_build_time_s=(
            "hierarchical_build_time_s",
            "sum",
        ),
    )
    .reset_index()
)

In [94]:
storage_summary[
    "key_multiplier"
] = (
    storage_summary[
        "hierarchical_keys"
    ]
    / storage_summary[
        "fixed_keys"
    ]
)

storage_summary[
    "membership_multiplier"
] = (
    storage_summary[
        "hierarchical_memberships"
    ]
    / storage_summary[
        "fixed_memberships"
    ]
)

storage_summary

,zoom,fixed_keys,fixed_memberships,fixed_build_time_s,hierarchical_keys,hierarchical_memberships,hierarchical_build_time_s,key_multiplier,membership_multiplier
0,23,29038,33393,0.448194,552580,825112,4.462256,19.029547,24.709131
1,24,58967,60711,0.410720,552580,825112,4.462256,9.371004,13.590816


In [95]:
storage_summary.to_csv(
    STORAGE_SUMMARY_OUTPUT,
    index=False,
)

In [96]:
for _, row in summary.iterrows():

    print()
    print(
        "=" * 60
    )

    print(
        f"ZOOM {int(row['zoom'])}"
    )

    print(
        "=" * 60
    )

    print(
        "Candidate agreement:",
        f"{row['agreement_rate']:.3f}",
    )

    print(
        "Fixed cells:",
        f"{row['mean_fixed_cells']:.1f}",
    )

    print(
        "Hierarchical cells:",
        f"{row['mean_hierarchical_cells']:.1f}",
    )

    print(
        "Compression:",
        f"{row['mean_compression_ratio']:.2f}x",
    )

    print(
        "Fixed precision:",
        f"{row['fixed_precision']:.4f}",
    )

    print(
        "Hier precision:",
        f"{row['hierarchical_precision']:.4f}",
    )

    print(
        "Fixed recall:",
        f"{row['fixed_recall']:.4f}",
    )

    print(
        "Hier recall:",
        f"{row['hierarchical_recall']:.4f}",
    )

    print(
        "Median fixed query:",
        f"{row['median_fixed_ms']:.3f} ms",
    )

    print(
        "Median hierarchical query:",
        f"{row['median_hierarchical_ms']:.3f} ms",
    )

    print(
        "Median speedup:",
        f"{row['median_query_speedup']:.2f}x",
    )

    print(
        "p95 fixed query:",
        f"{row['p95_fixed_ms']:.3f} ms",
    )

    print(
        "p95 hierarchical query:",
        f"{row['p95_hierarchical_ms']:.3f} ms",
    )


ZOOM 23
Candidate agreement: 1.000
Fixed cells: 239.9
Hierarchical cells: 16.2
Compression: 16.65x
Fixed precision: 0.9394
Hier precision: 0.9394
Fixed recall: 1.0000
Hier recall: 1.0000
Median fixed query: 0.984 ms
Median hierarchical query: 0.081 ms
Median speedup: 13.71x
p95 fixed query: 1.657 ms
p95 hierarchical query: 0.140 ms

ZOOM 24
Candidate agreement: 1.000
Fixed cells: 901.3
Hierarchical cells: 24.4
Compression: 40.06x
Fixed precision: 0.9592
Hier precision: 0.9592
Fixed recall: 1.0000
Hier recall: 1.0000
Median fixed query: 3.580 ms
Median hierarchical query: 0.116 ms
Median speedup: 39.98x
p95 fixed query: 6.300 ms
p95 hierarchical query: 0.167 ms
